In [2]:
import pandas as pd
import os

DATA_DIR = "data/offense_analysis"

# ---------------------------------------------------------------------------
# Helper: read a two-row-header CSV and flatten to single-level column names
# ---------------------------------------------------------------------------
def read_nfl_csv(path):
    df = pd.read_csv(path, header=[0, 1])

    # Flatten MultiIndex columns: keep only the lower-level name, which is the
    # actual stat label. The upper-level (BASIC, BASE, ANALYTICS, etc.) is just
    # a section header and adds noise when prepended.
    df.columns = [col[1].strip() for col in df.columns]

    # Standardise the team-name column (always called "Tm" in the lower header)
    df = df.rename(columns={"Tm": "team"})

    # Drop the rank column — it's positional, not meaningful as a join key
    df = df.drop(columns=["Rk"], errors="ignore")

    return df


# ---------------------------------------------------------------------------
# File manifest: (filename_stem, prefix_to_apply, year)
# The prefix is what gets prepended to every non-key column.
# ---------------------------------------------------------------------------
FILES = [
    # DVOA
    ("dvoa_offense_2024",          "dvoa",                   2024),
    ("dvoa_offense_2025",          "dvoa",                   2025),
    # Passing
    ("passing_base_2024",          "passing_base",           2024),
    ("passing_base_2025",          "passing_base",           2025),
    ("passing_coverage_2024",      "passing_coverage",       2024),
    ("passing_efficiency_2024",    "passing_efficiency",     2024),
    ("passing_efficiency_2025",    "passing_efficiency",     2025),
    ("passing_pressure_2025",      "passing_pressure",       2025),
    ("passing_redzone_2024",       "passing_redzone",        2024),
    ("passing_redzone_2025",       "passing_redzone",        2025),
    ("passing_tendencies_2024",    "passing_tendencies",     2024),
    ("passing_tendencies_2025",    "passing_tendencies",     2025),
    # Receiving
    ("recieving_efficiency_2024",  "receiving_efficiency",   2024),   # note the typo in filename
    ("receiving_efficiency_2025",  "receiving_efficiency",   2025),
    # Rushing
    ("rushing_base_2024",          "rushing_base",           2024),
    ("rushing_base_2025",          "rushing_base",           2025),
    ("rushing_efficiency_2024",    "rushing_efficiency",     2024),
    ("rushing_efficiency_2025",    "rushing_efficiency",     2025),
    ("rushing_types_2024",         "rushing_types",          2024),
    ("rushing_types_2025",         "rushing_types",          2025),
]

# Columns that are join keys — never prefixed
KEY_COLS = {"team", "G"}

# ---------------------------------------------------------------------------
# Load, tag, and prefix each file
# ---------------------------------------------------------------------------
def load_file(stem, prefix, year):
    path = os.path.join(DATA_DIR, f"{stem}.csv")
    df = read_nfl_csv(path)
    df["year"] = year

    # Prefix every column that isn't a key
    rename_map = {
        col: f"{prefix}_{col}"
        for col in df.columns
        if col not in KEY_COLS and col != "year"
    }
    df = df.rename(columns=rename_map)

    return df


# ---------------------------------------------------------------------------
# Build per-year stacks, then join across tables
# ---------------------------------------------------------------------------
def build_master():
    # Group files by year so we can join within each year first
    by_year = {2024: [], 2025: []}

    for stem, prefix, year in FILES:
        df = load_file(stem, prefix, year)
        by_year[year].append(df)

    year_dfs = []
    for year, dfs in by_year.items():
        # Start from the DVOA table (first entry for each year)
        master = dfs[0]
        for df in dfs[1:]:
            # Outer join so no team is silently dropped if a table is missing it
            shared_cols = list(KEY_COLS & set(df.columns)) + ["year"]
            master = master.merge(df, on=shared_cols, how="outer")

        year_dfs.append(master)

    # Stack both years into one dataframe
    combined = pd.concat(year_dfs, ignore_index=True)

    # Clean up: sort by year then team for readability
    combined = combined.sort_values(["year", "team"]).reset_index(drop=True)

    # Remove any fully-duplicate columns that crept in from overlapping
    # BASE stats repeated across files (e.g. ATT, COM, YDS appear in both
    # passing_base and passing_efficiency). Keep the first occurrence.
    combined = combined.loc[:, ~combined.columns.duplicated(keep="first")]

    return combined


In [3]:
df = build_master()

In [6]:
list(df.columns)

['team',
 'G',
 'dvoa_DVOA',
 'dvoa_Weighted DVOA',
 'dvoa_Passing DVOA',
 'dvoa_Unadjusted Passing DVOA',
 'dvoa_Rushing DVOA',
 'dvoa_Unadjusted Rushing DVOA',
 'year',
 'passing_base_SNP',
 'passing_base_ATT/G',
 'passing_base_DB',
 'passing_base_ATT',
 'passing_base_COM',
 'passing_base_COM%',
 'passing_base_YDS',
 'passing_base_YPA',
 'passing_base_TD',
 'passing_base_INT',
 'passing_base_FD',
 'passing_base_300Yd',
 'passing_base_EPA',
 'passing_base_EPA/DB',
 'passing_base_DVOA',
 'passing_coverage_ATT',
 'passing_coverage_COM',
 'passing_coverage_COM%',
 'passing_coverage_YDS',
 'passing_coverage_YPA',
 'passing_coverage_TD',
 'passing_coverage_SCK',
 'passing_coverage_SAK%',
 'passing_coverage_aSAK%',
 'passing_coverage_AVSACK',
 'passing_coverage_AVSACK%',
 'passing_coverage_QBSACK',
 'passing_coverage_QBSACK%',
 'passing_coverage_PRES',
 'passing_coverage_PRES%',
 'passing_coverage_CP',
 'passing_coverage_CP%',
 'passing_coverage_TTP',
 'passing_coverage_TTT',
 'passing_cove

In [ ]:
passing_coverage_SAK%
passing_coverage_PRES%
passing_coverage_SAK% - passing_coverage_PRES%
passing_efficiency_TOW%
passing_efficiency_ACC%
passing_efficiency_EXP%
passing_redzone_RZCM%